## Incremental sharded recovery + rolling checkpoints

`latest.pt` is atomically replaced after each completed epoch; `best.pt` is updated only when validation improves. The publisher creates **new, small Kaggle Dataset shards** containing only new or changed recovery files. It does not create a new version of a growing Dataset and does not copy old checkpoints into every publish.

Completed runs retain `result.json` metadata; only unfinished runs retain resumable checkpoints. KD teacher logits and selected-best post-hoc arrays are kept when needed. Upload staging uses hardlinks when possible, flat top-level files, and `--dir-mode skip` to avoid creating a second large ZIP/TAR on the HDD. Shards are split by file count and a soft size target.

**Important:** Kaggle does not automatically attach newly created Datasets to future sessions. Attach every `cs781-rec-shard-*` Dataset you need to restore. Creating more private Datasets does not bypass Kaggle's aggregate private-Dataset quota. Publishing fails safely: keep local checkpoints until uploads are verified.


# CS781 Phase 3 — Rhetorical Role Prediction (CL + IT): training pipeline

**What this notebook does:** loads the Phase 3 data, builds sentence-only and context-aware inputs, fine-tunes your Phase 1 language models with
six adaptation techniques (full FT, LoRA/rsLoRA, **LoRMA+/LoRMAπ**, IA³, QLoRA-style NF4, prompt tuning) plus **knowledge distillation**,
logs everything (macro-F1/P/R, trainable params, model size, GPU memory, time) to `ledger.csv`, and writes `prediction.json` + `submission.zip`.

**Run order:** 1 setup → 2 checkpoint inventory → 3 data → 4 tokenizer/examples → 5 stats → 6 metrics → 7 models → 8 trainer → **9 smoke tests (must pass)** → 10 plan → 11 RUN A → 12 RUN B → 13 KD → 14 summary → 15 post-hoc → 16 submission.

**Read before running**
* The dataset is gated, so cell 3 auto-detects the JSON layout and prints it. If it fails, paste the printed schema and set `CUSTOM_PARSER`.
* Only the GPT-2-style Phase 1 checkpoint has a loader. Cell 2 lists every checkpoint with a family guess; the other three architectures need a loader in cell 7 (send the cell 2 output).
* Check the output format of cell 16 against the official `prediction.json` schema (`COMBINE_MODE`).
* Dev has only a few documents per domain, so differences below ~0.02 macro-F1 are noise. Trust results that agree across CL and IT.


## Checkpointing and recovery (updated)
- Every completed training epoch writes an atomic `latest.pt` checkpoint containing model, optimizer, scheduler, AMP scaler, epoch, early-stopping state, and RNG states.
- `best.pt` stores the best validation macro-F1 model for inference.
- If an experiment is interrupted, rerun its cell: it resumes from `latest.pt` when available. Completed experiments with `result.json` are skipped unless `force=True`.
- The final cell creates `/kaggle/working/phase3_recovery.zip`. Download it or publish it as a dedicated Kaggle Dataset to make it durable outside the current session.
- This checkpointing update does not add missing Phase 1 architectures. The architecture-coverage requirement remains unresolved until those architectures are registered and tested.


In [ ]:
# ===== CELL 6 — REGISTER THE CORRECT PHASE 1 CHECKPOINT =====
# The architecture cell later in this notebook registers this exact checkpoint.
print("Target checkpoint: /kaggle/input/datasets/oipranesh/cs781-phase-1-checkpoints/checkpoint_latest.pt")


In [ ]:
# ================= CELL 0 — optional Kaggle secrets =================
# Do not require unrelated Kaggle API secrets just to run the notebook.
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
    try:
        HF_TOKEN = _secrets.get_secret("HF_TOKEN")
    except Exception:
        HF_TOKEN = None
except Exception:
    pass

print("HF_TOKEN secret available:", bool(HF_TOKEN))


In [ ]:
# ================= CELL 0b — SHARDED, INCREMENTAL KAGGLE RECOVERY =================
# Each publish creates a NEW small Dataset shard containing only files that are
# new/changed since the previous published shard. Files are flattened to top-level
# names and uploaded with --dir-mode skip, avoiding an extra ZIP/TAR on the HDD.
# IMPORTANT: attach every recovery shard to future notebook sessions.
import os, subprocess, shutil, json, time
from pathlib import Path

try:
    from kaggle_secrets import UserSecretsClient
    _ks = UserSecretsClient()
    for _secret_name in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
        if not os.environ.get(_secret_name):
            try: os.environ[_secret_name] = _ks.get_secret(_secret_name)
            except Exception: pass
except Exception:
    pass

KAGGLE_OWNER = os.environ.get("KAGGLE_USERNAME", "armankrsingh").strip().lower()
RECOVERY_SHARD_PREFIX = "cs781-rec-shard"
# Used for compatibility with the existing end-of-experiment hooks.
KAGGLE_DATASET_SLUG = f"{KAGGLE_OWNER}/{RECOVERY_SHARD_PREFIX}-series"
os.environ["KAGGLE_DATASET_SLUG"] = KAGGLE_DATASET_SLUG
PUBLISH_EVERY_N_EPOCHS = 0
PUBLISH_AT_EACH_EXPERIMENT_END = True
KD_TEACHER_RUNS = ("E0b_full_ctx", "E2_lorma_plus")
DURABLE_PUBLISH_ENABLED = bool(os.environ.get("KAGGLE_USERNAME")) and bool(os.environ.get("KAGGLE_KEY"))

if DURABLE_PUBLISH_ENABLED:
    print("Incremental sharded Kaggle publishing: ENABLED")
    print("New Dataset shards are created only when there are new/changed recovery files.")
    print("No old checkpoint collection is copied into each new shard.")
else:
    print("Kaggle publishing is not configured; local rolling checkpoints still work.")


In [ ]:
# ================= CELL 1 — environment, paths, helpers =================
import os, sys, json, math, time, random, re, copy, gc, glob, zipfile, shutil
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from huggingface_hub import snapshot_download, HfApi

SEED = 42

def seed_all(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device,
      torch.cuda.get_device_name(0) if device.type == "cuda"
      else "(no GPU: training will be slow)")

WORK = Path("/kaggle/working/phase3_rr")
(WORK / "runs").mkdir(parents=True, exist_ok=True)
LEDGER = WORK / "ledger.csv"



# ---------------- INCREMENTAL SHARDED KAGGLE DATASET PUBLISHER ----------------
import hashlib
PUBLISH_DIR = Path("/kaggle/working/phase3_publish_shard")
MAX_FILES_PER_SHARD = 40       # keep below Kaggle's 50 top-level-file limit
MAX_BYTES_PER_SHARD = 10 * 1024**3  # soft batching target; a single larger file is its own shard
KD_TEACHER_RUNS = tuple(globals().get("KD_TEACHER_RUNS", ("E0b_full_ctx", "E2_lorma_plus")))
SHARD_INPUT_ROOT = Path("/kaggle/input/datasets")


def _completed_run(run_dir):
    p = run_dir / "result.json"
    if not p.is_file(): return False
    try:
        obj = json.loads(p.read_text(encoding="utf-8"))
        return isinstance(obj, dict) and bool(obj.get("name")) and bool(obj.get("domain"))
    except Exception:
        return False


def _selected_best_run_name():
    if not LEDGER.exists(): return None
    try:
        df = pd.read_csv(LEDGER)
        if df.empty or not {"name", "domain", "dev_macro_f1"}.issubset(df.columns): return None
        if "merge_dev" in df.columns:
            df = df[~df["merge_dev"].astype(str).str.lower().isin(["true", "1"])]
        df["dev_macro_f1"] = pd.to_numeric(df["dev_macro_f1"], errors="coerce")
        agg = df.groupby("name", dropna=True)["dev_macro_f1"].mean().dropna()
        return str(agg.idxmax()) if not agg.empty else None
    except Exception as e:
        print("[shard backup] Could not select best run:", repr(e)); return None


def _sha256(path, chunk=8 * 1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""): h.update(block)
    return h.hexdigest()


def _attached_shard_roots():
    roots = []
    if not SHARD_INPUT_ROOT.exists(): return roots
    for owner_dir in SHARD_INPUT_ROOT.iterdir():
        if not owner_dir.is_dir(): continue
        for ds in owner_dir.iterdir():
            if ds.is_dir() and ds.name.startswith(RECOVERY_SHARD_PREFIX):
                roots.append(ds)
    return sorted(roots)


def _published_index():
    # Each shard manifest lists original relative path, checksum, flat filename,
    # and publication timestamp. Prefer the newest entry for a path.
    index = {}
    for root in _attached_shard_roots():
        mp = root / "shard_manifest.json"
        if not mp.is_file(): continue
        try: obj = json.loads(mp.read_text(encoding="utf-8"))
        except Exception: continue
        ts = float(obj.get("published_unix", 0))
        for item in obj.get("files", []):
            rel = item.get("relative_path")
            if rel and (rel not in index or ts >= index[rel].get("published_unix", 0)):
                index[rel] = {"sha256": item.get("sha256"), "shard": root.name,
                              "flat_name": item.get("flat_name"), "published_unix": ts}
    return index


def _recovery_inventory():
    """List only artifacts needed for resume, completion metadata, KD and post-hoc."""
    files = {}
    runs_src = WORK / "runs"
    selected_best = _selected_best_run_name()
    if runs_src.exists():
        for run_dir in sorted(runs_src.iterdir()):
            if not run_dir.is_dir(): continue
            done = _completed_run(run_dir)
            wanted = set()
            if (run_dir / "result.json").is_file(): wanted.add("result.json")
            if not done:
                wanted.update(n for n in ("latest.pt", "best.pt") if (run_dir / n).is_file())
            if any(run_dir.name == f"{name}_{dom}" for name in KD_TEACHER_RUNS for dom in ("CL", "IT")):
                if (run_dir / "train_logits.npy").is_file(): wanted.add("train_logits.npy")
            if selected_best and run_dir.name in (f"{selected_best}_CL", f"{selected_best}_IT"):
                wanted.update(n for n in ("train_logits.npy", "dev_logits.npy", "test_logits.npy",
                    "train_emb.npy", "dev_emb.npy", "test_emb.npy", "train_y.npy") if (run_dir / n).is_file())
            for name in wanted:
                p = run_dir / name
                files[p.relative_to(WORK).as_posix()] = p
    for name in ("ledger.csv", "prediction.json", "prediction_CL.json", "prediction_IT.json"):
        p = WORK / name
        if p.is_file(): files[p.relative_to(WORK).as_posix()] = p
    return files


def _flat_name(relative_path):
    # Flat top-level files let the CLI use --dir-mode skip, avoiding tar/zip staging.
    safe = re.sub(r"[^A-Za-z0-9._-]+", "_", relative_path.replace("/", "__"))
    if len(safe) > 180:
        safe = safe[:130] + "__" + hashlib.sha1(relative_path.encode()).hexdigest()[:16]
    return "f__" + safe


def _make_shard_dir(batch, batch_no, published_index, reason):
    stamp = time.strftime("%Y%m%d-%H%M%S", time.gmtime())
    unique = int(time.time() * 1000) % 100000
    slug = f"{RECOVERY_SHARD_PREFIX}-{stamp}-{unique:05d}-{batch_no:02d}".lower()
    # Kaggle slug max is limited; this pattern stays comfortably below it.
    if PUBLISH_DIR.exists(): shutil.rmtree(PUBLISH_DIR)
    PUBLISH_DIR.mkdir(parents=True, exist_ok=True)
    entries = []
    for rel, src in batch:
        flat = _flat_name(rel)
        dst = PUBLISH_DIR / flat
        # Hardlinks avoid making a second full copy on the same filesystem.
        try: os.link(src, dst)
        except OSError: shutil.copy2(src, dst)
        digest = _sha256(src)
        entries.append({"relative_path": rel, "flat_name": flat, "sha256": digest,
                        "size_bytes": src.stat().st_size})
        published_index[rel] = {"sha256": digest, "shard": slug, "flat_name": flat,
                                "published_unix": time.time()}
    manifest = {
        "project": "CS781 Phase 3 recovery shard", "dataset_slug": f"{KAGGLE_OWNER}/{slug}",
        "published_unix": time.time(), "reason": reason, "files": entries,
        "file_count": len(entries), "policy": "Incremental changed/new files only; completed runs keep metadata; checkpoints only for unfinished runs."
    }
    (PUBLISH_DIR / "shard_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    metadata = {"title": f"CS781 Recovery {stamp} {batch_no:02d}", "id": f"{KAGGLE_OWNER}/{slug}",
                "licenses": [{"name": "CC0-1.0"}], "isPrivate": True,
                "description": "Incremental CS781 recovery shard. Attach all CS781 recovery shards to restore the full recovery history."}
    (PUBLISH_DIR / "dataset-metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
    return slug, entries


def publish_recovery_dataset(reason="periodic"):
    """Publish changed files as one or more NEW Dataset shards; never version old shards."""
    if not DURABLE_PUBLISH_ENABLED:
        print("[shard backup] Skipped: Kaggle credentials not configured."); return False
    cli = shutil.which("kaggle")
    if cli is None:
        print("[shard backup] Installing Kaggle CLI...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"], check=True)
        cli = shutil.which("kaggle")
    if cli is None: raise RuntimeError("Kaggle CLI was not installed correctly.")

    env = os.environ.copy()
    env["KAGGLE_USERNAME"] = os.environ["KAGGLE_USERNAME"]
    env["KAGGLE_KEY"] = os.environ["KAGGLE_KEY"]
    inventory = _recovery_inventory()
    prior = _published_index()
    changed = []
    for rel, path in sorted(inventory.items()):
        # Always checksum local candidate to detect changed checkpoints/results.
        digest = _sha256(path)
        if prior.get(rel, {}).get("sha256") != digest:
            changed.append((rel, path))
    if not changed:
        print("[shard backup] Nothing new or changed to publish.")
        if PUBLISH_DIR.exists(): shutil.rmtree(PUBLISH_DIR, ignore_errors=True)
        return True

    # Batches stay below 40 payload files and a soft 10 GiB target. A single large
    # checkpoint may exceed the target but is still isolated in its own shard.
    batches, batch, nbytes = [], [], 0
    for item in changed:
        size = item[1].stat().st_size
        if batch and (len(batch) >= MAX_FILES_PER_SHARD or nbytes + size > MAX_BYTES_PER_SHARD):
            batches.append(batch); batch, nbytes = [], 0
        batch.append(item); nbytes += size
    if batch: batches.append(batch)

    print(f"[shard backup] {len(changed)} new/changed files -> {len(batches)} new Dataset shard(s).")
    uploaded = []
    try:
        for i, items in enumerate(batches, 1):
            slug, entries = _make_shard_dir(items, i, prior, reason)
            free = shutil.disk_usage("/kaggle/working").free
            payload = sum(x[1].stat().st_size for x in items)
            print(f"[shard backup] Creating {KAGGLE_OWNER}/{slug}: {len(items)} files, {payload/1024**3:.2f} GiB; free disk {free/1024**3:.2f} GiB")
            # Flat top-level files and --dir-mode skip prevent CLI from building a
            # second giant tar/zip archive on the local HDD.
            cmd = [cli, "datasets", "create", "-p", str(PUBLISH_DIR), "-t", "-r", "skip"]
            result = subprocess.run(cmd, env=env, capture_output=True, text=True)
            out = (result.stdout or "") + "\n" + (result.stderr or "")
            if result.stdout: print(result.stdout[-3000:])
            if result.stderr: print(result.stderr[-3000:])
            if result.returncode != 0 or not any(x in out.lower() for x in ("created", "success", "dataset")):
                raise RuntimeError(f"Shard creation not confirmed for {KAGGLE_OWNER}/{slug}. Do not delete local checkpoints. CLI output:\n{out[-5000:]}")
            uploaded.append(f"{KAGGLE_OWNER}/{slug}")
            # Clear staging links immediately; hardlinks mean source artifacts remain.
            shutil.rmtree(PUBLISH_DIR, ignore_errors=True)
    finally:
        shutil.rmtree(PUBLISH_DIR, ignore_errors=True)
    print("[shard backup] Submitted shards:", uploaded)
    print("[shard backup] Attach all listed recovery shards as notebook inputs in future sessions.")
    print("[shard backup] Keep local checkpoints until you verify each Dataset page and input mount.")
    return True

def find_files(names, roots=("/kaggle/input", "/kaggle/working", "/content", ".")):
    found = {}
    for root in roots:
        if not os.path.exists(root):
            continue
        for n in names:
            if n in found:
                continue
            hits = sorted(glob.glob(f"{root}/**/{n}", recursive=True))
            if hits:
                found[n] = hits[0]
    return found

NEED = [
    "CL_train.json", "CL_dev.json", "CL_test.json",
    "IT_train.json", "IT_dev.json", "IT_test.json",
    "label_vocab.json"
]

paths = find_files(NEED)
missing = [n for n in NEED if n not in paths]

if missing:
    print("Missing locally:", missing)
    print("Trying Hugging Face dataset download...")

    token = os.environ.get("HF_TOKEN") or HF_TOKEN

    # Kaggle Secrets fallback
    if not token:
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret("HF_TOKEN")
        except Exception:
            token = None

    if not token:
        raise RuntimeError(
            "HF_TOKEN was not found. In Kaggle, open Add-ons > Secrets, "
            "add a secret named HF_TOKEN with your Hugging Face READ token, "
            "and enable notebook access. Also accept the dataset conditions "
            "on the Hugging Face dataset page. Alternatively, attach the "
            "Phase_3_RR dataset to this Kaggle notebook."
        )

    try:
        local = snapshot_download(
            repo_id="Exploration-Lab/CS-781-Capstone",
            repo_type="dataset",
            token=token,
            allow_patterns=["Phase_3_RR/**"],
            local_dir=str(WORK / "hf_data")
        )
    except Exception as e:
        raise RuntimeError(
            "Hugging Face download failed. Check that you have accepted "
            "the dataset conditions and that HF_TOKEN is a valid token "
            "with read access. Original error: " + repr(e)
        ) from e

    paths = find_files(NEED, roots=(local, str(WORK / "hf_data")))
    missing = [n for n in NEED if n not in paths]

    if missing:
        # Fallback search if the files were placed in nested folders
        paths.update(find_files(missing, roots=(str(WORK / "hf_data"), local)))
        missing = [n for n in NEED if n not in paths]

    if missing:
        raise FileNotFoundError(
            f"Download completed, but these files were not found: {missing}. "
            "Check the dataset's folder structure."
        )

for k, v in paths.items():
    print(f"{k:18s} {v}")

print("Cell 1 setup complete.")

# Load the discovered files once. The parser cell below will convert them into DOCS.
RAW = {name: json.loads(Path(path).read_text(encoding="utf-8")) for name, path in paths.items()}
print("Loaded raw JSON files:", {k: type(v).__name__ for k, v in RAW.items()})


In [ ]:
# ================= CELL 2 — inventory of every Phase 1 checkpoint (use this to fill the architecture registry) =================
def family_guess(keys):
    s = " ".join(keys[:400])
    if "attn.c_attn" in s: return "gpt2-style (HF GPT2: fused c_attn Conv1D)"
    if "self_attn.q_proj" in s and "gate_proj" in s: return "llama-style (q_proj/k_proj/v_proj + gated MLP)"
    if "A_log" in s or "mixer" in s or "in_proj" in s: return "state-space / mamba-style"
    if "rwkv" in s.lower() or "time_mix" in s: return "rwkv-style"
    if "encoder.layer" in s or "attention.self.query" in s: return "bert-style (bidirectional encoder)"
    return "unknown - paste the key names to the assistant"

CKPT_EXT = (".pt", ".pth", ".bin", ".ckpt")
cands = sorted(p for p in glob.glob("/kaggle/input/**/*", recursive=True) if p.endswith(CKPT_EXT) and os.path.getsize(p) > 5e6)
print(f"{len(cands)} candidate checkpoint files under /kaggle/input\n")
for p in cands:
    try:
        try: obj = torch.load(p, map_location="cpu", weights_only=False, mmap=True)
        except Exception: obj = torch.load(p, map_location="cpu", weights_only=False)
    except Exception as e:
        print(p, "-> load failed:", str(e)[:120], "\n"); continue
    sd = obj.get("model_state_dict", obj) if isinstance(obj, dict) else None
    print(p, f"({os.path.getsize(p)/2**20:.0f} MB)")
    if isinstance(obj, dict):
        print("   top-level keys:", [k for k in obj.keys()][:12])
        if "model_config" in obj: print("   model_config  :", obj["model_config"])
        for k in ("step", "epoch", "context_length"):
            if k in obj: print(f"   {k}: {obj[k]}")
    if isinstance(sd, dict) and sd and hasattr(next(iter(sd.values())), "numel"):
        keys = list(sd.keys()); n = sum(v.numel() for v in sd.values() if hasattr(v, "numel"))
        print(f"   tensors: {len(keys)} | params: {n/1e6:.1f}M | family guess: {family_guess(keys)}")
        print("   first keys:", keys[:3])
    print()
    del obj, sd; gc.collect()
print("Map each of the FOUR Phase 1 models to an entry in ARCHS (cell 7). If a family says 'unknown', send this output to get a loader.")


In [ ]:
# ============================================================
# RECOVERY — scan every attached shard; metadata first, checkpoints only as files
# Attach all Dataset inputs whose names start cs781-rec-shard-.
# ============================================================
from pathlib import Path
import os, shutil, json, time

WORK = Path("/kaggle/working/phase3_rr")
RUNS = WORK / "runs"
RUNS.mkdir(parents=True, exist_ok=True)
INPUT_ROOT = Path("/kaggle/input/datasets")
shard_roots = []
if INPUT_ROOT.exists():
    for owner_dir in INPUT_ROOT.iterdir():
        if owner_dir.is_dir():
            for ds in owner_dir.iterdir():
                if ds.is_dir() and ds.name.startswith("cs781-rec-shard-"):
                    shard_roots.append(ds)
legacy_candidates = [
    INPUT_ROOT / "armankrsingh" / "cs781-phase3-recovery-compact",
    INPUT_ROOT / "armankrsingh" / "cs781-phase3-recovery",
]
legacy = next((p for p in legacy_candidates if p.exists()), None)

# Map each original relative path to the newest attached shard copy, without loading
# checkpoint contents into RAM. Every shard's manifest gives the original path.
latest = {}
for root in sorted(shard_roots):
    mp = root / "shard_manifest.json"
    if not mp.is_file(): continue
    try: meta = json.loads(mp.read_text(encoding="utf-8"))
    except Exception as e:
        print("[recovery] Invalid shard manifest:", mp, repr(e)); continue
    ts = float(meta.get("published_unix", 0))
    for item in meta.get("files", []):
        rel, flat = item.get("relative_path"), item.get("flat_name")
        if not rel or not flat: continue
        src = root / flat
        if src.is_file() and (rel not in latest or ts >= latest[rel][0]):
            latest[rel] = (ts, src)

restored, unchanged, conflicts = [], [], []
for rel, (_ts, src) in sorted(latest.items()):
    dst = WORK / rel
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        # Don't overwrite a local file from this live session; latest shard remains
        # available as a source if a manual recovery is needed.
        unchanged.append(rel); continue
    try: os.link(src, dst)
    except Exception: shutil.copy2(src, dst)
    restored.append(rel)

# Compatibility restore for the previous nested compact Dataset format.
if legacy is not None:
    allowed = {"latest.pt", "best.pt", "result.json", "train_logits.npy", "dev_logits.npy", "test_logits.npy", "train_emb.npy", "dev_emb.npy", "test_emb.npy", "train_y.npy"}
    for src in sorted((legacy / "runs").rglob("*")) if (legacy / "runs").exists() else []:
        if not src.is_file() or src.name not in allowed: continue
        rel = src.relative_to(legacy).as_posix()
        dst = WORK / rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        if not dst.exists():
            try: os.link(src, dst)
            except Exception: shutil.copy2(src, dst)
            restored.append(rel)
    for name in ("ledger.csv", "prediction.json", "prediction_CL.json", "prediction_IT.json"):
        src, dst = legacy / name, WORK / name
        if src.is_file() and not dst.exists():
            try: os.link(src, dst)
            except Exception: shutil.copy2(src, dst)
            restored.append(name)

if not shard_roots and legacy is None:
    print("[recovery] No recovery shard or legacy recovery Dataset is attached.")
    print("Attach all available cs781-rec-shard-* Datasets under Notebook > Add Input.")
else:
    print("[recovery] Attached shards:", [p.name for p in shard_roots])
    print("[recovery] Legacy source:", legacy)
print(f"[recovery] Restored: {len(restored)} | already present: {len(unchanged)}")
print("[recovery] Run status check uses result.json metadata only; checkpoint tensors are not loaded.")
for run_dir in sorted(p for p in RUNS.iterdir() if p.is_dir()):
    rp = run_dir / "result.json"
    if rp.exists():
        try:
            r = json.loads(rp.read_text(encoding="utf-8"))
            valid = isinstance(r, dict) and bool(r.get("name")) and bool(r.get("domain"))
            print(f"[DONE] {run_dir.name}: dev_macro_f1={r.get('dev_macro_f1', 'n/a')}" if valid else f"[INVALID RESULT] {run_dir.name}")
        except Exception as e: print(f"[INVALID RESULT] {run_dir.name}: {e}")
    else:
        print(f"[INCOMPLETE] {run_dir.name}: latest.pt={'yes' if (run_dir/'latest.pt').exists() else 'no'}")
print("Recovery finished; no training was started.")


In [ ]:
# ================= CELL 3 — PARSE PHASE 3 DATA (TRAIN / DEV / TEST) =================
# Supports the observed test schema {doc_id: [sentence, ...]} and common labeled schemas.

LABELS_FALLBACK = [
    "Fact", "Issue", "ArgumentPetitioner", "ArgumentRespondent", "Statute",
    "Dissent", "PrecedentReliedUpon", "PrecedentNotReliedUpon",
    "PrecedentOverruled", "RulingByLowerCourt", "RatioOfTheDecision",
    "RulingByPresentCourt", "None"
]

def _norm_key(k):
    return str(k).strip().lower().replace("-", "_").replace(" ", "_")

def _find_sentence_list(payload):
    if isinstance(payload, list):
        if all(isinstance(x, str) for x in payload):
            return payload
        if payload and all(isinstance(x, dict) for x in payload):
            out = []
            for item in payload:
                sent = None
                for k, v in item.items():
                    if _norm_key(k) in {"text", "sentence", "sent", "content"} and isinstance(v, str):
                        sent = v
                        break
                if sent is None:
                    out = []
                    break
                out.append(sent)
            if out:
                return out
        for item in payload:
            got = _find_sentence_list(item)
            if got is not None:
                return got
        return None

    if isinstance(payload, dict):
        # Prefer conventional keys first.
        for k, v in payload.items():
            if _norm_key(k) in {"text", "texts", "sentences", "sentence", "sents", "content"}:
                if isinstance(v, list) and all(isinstance(x, str) for x in v):
                    return v
                if isinstance(v, str):
                    return [v]
        # Search nested values, but don't treat a labels list as sentence text.
        for k, v in payload.items():
            if _norm_key(k) in {"labels", "label", "y", "tags"}:
                continue
            if isinstance(v, (dict, list)):
                got = _find_sentence_list(v)
                if got is not None:
                    return got
    return None

def _find_label_list(payload):
    if isinstance(payload, dict):
        for k, v in payload.items():
            if _norm_key(k) in {"labels", "label", "tags", "roles", "y"} and isinstance(v, list):
                return v
        for k, v in payload.items():
            if isinstance(v, (dict, list)):
                got = _find_label_list(v)
                if got is not None:
                    return got
    elif isinstance(payload, list):
        if payload and all(isinstance(x, dict) for x in payload):
            vals = []
            for item in payload:
                lab = None
                for k, v in item.items():
                    if _norm_key(k) in {"label", "labels", "tag", "role"} and isinstance(v, str):
                        lab = v
                        break
                if lab is None:
                    vals = []
                    break
                vals.append(lab)
            if vals:
                return vals
        for item in payload:
            if isinstance(item, (dict, list)):
                got = _find_label_list(item)
                if got is not None:
                    return got
    return None

def parse_split(obj, style=None):
    if isinstance(obj, dict):
        items = list(obj.items())
    elif isinstance(obj, list):
        items = [(str(i), x) for i, x in enumerate(obj)]
    else:
        raise ValueError(f"Unsupported split JSON type: {type(obj).__name__}")

    docs = []
    for did, payload in items:
        # The test files were confirmed to map document IDs directly to sentence strings.
        if isinstance(payload, list) and all(isinstance(x, str) for x in payload):
            docs.append({"id": str(did), "sents": payload, "labels": None, "setter": None})
            continue

        sents = _find_sentence_list(payload)
        labels = _find_label_list(payload)
        if sents is None:
            raise ValueError(
                f"{did}: cannot find a sentence list. Payload type={type(payload).__name__}; "
                f"keys={list(payload.keys())[:30] if isinstance(payload, dict) else 'n/a'}"
            )
        if labels is not None:
            labels = [str(x) for x in labels]
            if len(labels) != len(sents):
                raise ValueError(f"{did}: {len(sents)} sentences but {len(labels)} labels")
        docs.append({"id": str(did), "sents": [str(x) for x in sents],
                     "labels": labels, "setter": None})
    return docs, style

# Load label vocabulary if it is a recognizable format; otherwise use the official label names.
_vocab_obj = RAW.get("label_vocab.json")
LABELS = None
if isinstance(_vocab_obj, list) and all(isinstance(x, str) for x in _vocab_obj):
    LABELS = _vocab_obj
elif isinstance(_vocab_obj, dict):
    for key in ("labels", "label_vocab", "id2label", "label2id"):
        value = _vocab_obj.get(key)
        if isinstance(value, list) and all(isinstance(x, str) for x in value):
            LABELS = value
            break
        if isinstance(value, dict) and value:
            try:
                LABELS = [k for k, v in sorted(value.items(), key=lambda kv: int(kv[1]))]
            except Exception:
                LABELS = list(value.keys())
            break
if not LABELS:
    LABELS = LABELS_FALLBACK
LABEL2ID = {lab: i for i, lab in enumerate(LABELS)}
STYLE = {"CL": None, "IT": None}

DOCS = {"CL": {}, "IT": {}}
TEST_OBJ = {}
for dom in ("CL", "IT"):
    for split in ("train", "dev"):
        DOCS[dom][split], _ = parse_split(RAW[f"{dom}_{split}.json"])
        if not DOCS[dom][split]:
            raise ValueError(f"{dom} {split} split is empty")
        for d in DOCS[dom][split]:
            if d["labels"] is None:
                raise ValueError(f"{dom} {split} document {d['id']} has no labels")
            bad = sorted(set(d["labels"]) - set(LABELS))
            if bad:
                raise ValueError(f"{dom} {split} document {d['id']} has labels absent from vocabulary: {bad}")
    TEST_OBJ[dom] = RAW[f"{dom}_test.json"]
    DOCS[dom]["test"], _ = parse_split(TEST_OBJ[dom])
    print(dom, {sp: len(DOCS[dom][sp]) for sp in ("train", "dev", "test")})
    print("  sample sentence counts:", [len(d["sents"]) for d in DOCS[dom]["test"][:2]])

print("Labels:", LABELS)
print("DOCS and LABEL2ID are ready.")


# Validate schema and labels before training.
assert set(("CL", "IT")).issubset(DOCS)
for _dom in ("CL", "IT"):
    for _split in ("train", "dev"):
        assert DOCS[_dom][_split], f"Empty {_dom}/{_split}"
        for _d in DOCS[_dom][_split]:
            assert _d["labels"] is not None, f"Missing labels in {_dom}/{_split}/{_d['id']}"
            assert len(_d["sents"]) == len(_d["labels"]), f"Sentence/label mismatch in {_dom}/{_split}/{_d['id']}"
    for _d in DOCS[_dom]["test"]:
        assert _d["labels"] is None, f"Expected unlabeled test doc: {_d['id']}"
print("Train/dev/test schema validation passed.")


In [ ]:
# ================= CELL 5 — tokenizer, example builder (sentence-only vs context-aware), batching =================
from tokenizers import Tokenizer
_tok_files = find_files(["legal_bpe_32k.json"])
if "legal_bpe_32k.json" not in _tok_files:
    raise FileNotFoundError(
        "legal_bpe_32k.json was not found. Attach the Phase 1 tokenizer dataset "
        "to Kaggle or update tok_path to its actual path."
    )
tok_path = _tok_files["legal_bpe_32k.json"]
tokenizer = Tokenizer.from_file(tok_path)
print("tokenizer:", tok_path, "| vocab:", tokenizer.get_vocab_size())

_ENC = {}
def enc(text):
    r = _ENC.get(text)
    if r is None: r = _ENC[text] = tokenizer.encode(text, add_special_tokens=False).ids
    return r

MK = {"prev": enc(" <<previous>> "), "next": enc(" <<next>> "), "tgt": enc(" <<target>> ")}
SEP = enc(" | ")

def build_examples(docs, ctx, k_left=3, k_right=1, t_budget=96, l_budget=160, r_budget=64, max_sent=96):
    """One example per sentence. Context-aware input (for a causal LM the target comes LAST so it can attend to everything):
       <<previous>> left sentences <<next>> right sentences <<target>> target sentence."""
    exs = []
    for d in docs:
        toks = [enc(s)[:max_sent] for s in d["sents"]]
        n = len(toks)
        for i in range(n):
            tgt = toks[i][:t_budget]
            if ctx:
                left, right = [], []
                for j in range(max(0, i - k_left), i): left += toks[j] + SEP
                for j in range(i + 1, min(n, i + 1 + k_right)): right += toks[j] + SEP
                ids = MK["prev"] + left[-l_budget:] + MK["next"] + right[:r_budget] + MK["tgt"] + tgt
            else:
                ids = list(tgt)
            if not ids: ids = [0]
            rel = i / max(n - 1, 1)
            y = LABEL2ID[d["labels"][i]] if d["labels"] is not None and d["labels"][i] is not None else -1
            exs.append(dict(ids=ids, pos=[rel, rel * rel], y=y, doc=d["id"], i=i))
    return exs

def make_batches(exs, bs, shuffle, rng):
    idx = list(range(len(exs)))
    if shuffle: rng.shuffle(idx)
    batches = []
    for c in range(0, len(idx), bs * 50):
        ch = sorted(idx[c:c + bs * 50], key=lambda j: len(exs[j]["ids"]))
        batches += [ch[k:k + bs] for k in range(0, len(ch), bs)]
    if shuffle: rng.shuffle(batches)
    return batches

def collate(exs, bidx):
    import torch
    L = max(len(exs[j]["ids"]) for j in bidx)
    ids = torch.zeros((len(bidx), L), dtype=torch.long); mask = torch.zeros_like(ids)
    for r, j in enumerate(bidx):
        n = len(exs[j]["ids"]); ids[r, :n] = torch.tensor(exs[j]["ids"]); mask[r, :n] = 1
    pos = torch.tensor([exs[j]["pos"] for j in bidx], dtype=torch.float32)
    y = torch.tensor([exs[j]["y"] for j in bidx], dtype=torch.long)
    return ids, mask, pos, y


In [ ]:
# ================= CELL 4 — parser status =================
# Do not redefine parse_split here. Cell 3 is the single source of truth.
print("DOCS keys:", list(DOCS.keys()))
for _dom in ("CL", "IT"):
    print(_dom, {k: len(v) for k, v in DOCS[_dom].items()})


In [ ]:
# ================= CELL 7a — metrics + inference-time post-processing (numpy only) =================
def prf_report(y_true, y_pred, K):
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    cm = np.zeros((K, K), dtype=np.int64); np.add.at(cm, (y_true, y_pred), 1)
    tp = np.diag(cm).astype(float); fp = cm.sum(0) - tp; fn = cm.sum(1) - tp
    prec = np.divide(tp, tp + fp, out=np.zeros(K), where=(tp + fp) > 0)
    rec  = np.divide(tp, tp + fn, out=np.zeros(K), where=(tp + fn) > 0)
    f1   = np.divide(2 * prec * rec, prec + rec, out=np.zeros(K), where=(prec + rec) > 0)
    present = (cm.sum(0) + cm.sum(1)) > 0          # labels in gold or prediction = sklearn's default macro average
    return dict(macro_f1=float(f1[present].mean()), macro_p=float(prec[present].mean()), macro_r=float(rec[present].mean()),
                macro_f1_all=float(f1.mean()), acc=float(tp.sum() / cm.sum()), f1=f1, prec=prec, rec=rec,
                support=cm.sum(1), cm=cm, n_present=int(present.sum()))

def softmax_np(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True); e = np.exp(x); return e / e.sum(axis=axis, keepdims=True)

def transition_logprobs(docs, label2id, K, alpha=1.0):
    T = np.full((K, K), alpha); S = np.full(K, alpha)
    for d in docs:
        ys = [label2id[l] for l in d["labels"]]
        S[ys[0]] += 1
        for a, b in zip(ys[:-1], ys[1:]): T[a, b] += 1
    return np.log(T / T.sum(1, keepdims=True)), np.log(S / S.sum())

def viterbi_smooth(logp, logT, logS, lam):
    """argmax_y  sum_i logp[i,y_i] + lam * (logS[y_0] + sum_i logT[y_{i-1}, y_i])"""
    n, K = logp.shape
    if lam == 0 or n == 1: return logp.argmax(1)
    dp = np.zeros((n, K)); bp = np.zeros((n, K), dtype=int)
    dp[0] = logp[0] + lam * logS
    for i in range(1, n):
        cand = dp[i - 1][:, None] + lam * logT
        bp[i] = cand.argmax(0); dp[i] = cand.max(0) + logp[i]
    path = [int(dp[-1].argmax())]
    for i in range(n - 1, 0, -1): path.append(int(bp[i, path[-1]]))
    return np.array(path[::-1])

def group_by_doc(exs):
    g = {}
    for j, e in enumerate(exs): g.setdefault(e["doc"], []).append(j)
    return {d: sorted(ix, key=lambda j: exs[j]["i"]) for d, ix in g.items()}

def smooth_split(probs, exs, logT, logS, lam):
    pred = np.zeros(len(exs), dtype=int)
    for d, ix in group_by_doc(exs).items():
        pred[ix] = viterbi_smooth(np.log(np.clip(probs[ix], 1e-9, 1)), logT, logS, lam)
    return pred

def knn_interpolate(q_emb, tr_emb, tr_y, q_probs, K, k=10, tau=0.1, lam=0.5):
    """p = lam * p_model + (1-lam) * p_kNN  (neighbourhood-at-inference idea; neighbours come from TRAIN sentences only)"""
    qe = q_emb / (np.linalg.norm(q_emb, axis=1, keepdims=True) + 1e-9)
    te = tr_emb / (np.linalg.norm(tr_emb, axis=1, keepdims=True) + 1e-9)
    sims = qe @ te.T
    top = np.argpartition(-sims, kth=min(k, sims.shape[1] - 1), axis=1)[:, :k]
    out = np.zeros_like(q_probs)
    for r in range(len(qe)):
        s = sims[r, top[r]] / tau; w = softmax_np(s)
        np.add.at(out[r], tr_y[top[r]], w)
    return lam * q_probs + (1 - lam) * out

def validate_submission(obj, docs, label_set, style=None):
    """every document and sentence present, order preserved, exactly one valid label per sentence"""
    d2 = {d["id"]: d for d in docs}
    problems = []
    parsed, _ = parse_split(obj, style)
    if [d["id"] for d in parsed] != [d["id"] for d in docs]: problems.append("document ids / order differ from the test file")
    for p in parsed:
        ref = d2.get(p["id"])
        if ref is None: continue
        if p["sents"] != ref["sents"]: problems.append(f"{p['id']}: sentence text/order changed")
        if p["labels"] is None or len(p["labels"]) != len(ref["sents"]): problems.append(f"{p['id']}: label count mismatch"); continue
        bad = [l for l in p["labels"] if l not in label_set]
        if bad: problems.append(f"{p['id']}: invalid labels {set(bad)}")
    return problems


In [ ]:
# ================= CELL 6 — backbones, classifier, adapters (LoRA / rsLoRA / LoRMA+ / LoRMAπ / IA3 / QLoRA / prompt) =================
from transformers import GPT2Config, GPT2Model
from transformers.pytorch_utils import Conv1D

# ---- architecture registry: exact known 3,000-step Phase 1 checkpoint ----
PHASE1_CHECKPOINT = "/kaggle/input/datasets/oipranesh/cs781-phase-1-checkpoints/checkpoint_latest.pt"
if not os.path.isfile(PHASE1_CHECKPOINT):
    raise FileNotFoundError(
        f"Required 3,000-step checkpoint not found: {PHASE1_CHECKPOINT}. "
        "Attach the Kaggle dataset 'CS781 Phase 1 Checkpoints'."
    )
ARCHS = {"gpt2_a": {"path": PHASE1_CHECKPOINT, "family": "gpt2"}}
print("Registered architecture:", ARCHS)

def build_backbone(arch, bb_dropout=0.1, n_layers=None):
    """-> (backbone, hidden, causal, n_positions, family). n_layers: keep only the first n layers (student initialisation)."""
    spec = ARCHS[arch]
    if spec["family"] != "gpt2":
        raise NotImplementedError(f"No loader for family '{spec['family']}'. Run the inventory cell, paste its output, and add a loader here.")
    ck = torch.load(spec["path"], map_location="cpu", weights_only=False)
    cfg = ck.get("model_config")
    if not isinstance(cfg, dict):
        raise ValueError(f"Checkpoint has no model_config dictionary: {spec['path']}")
    vocab_size = cfg.get("vocab_size")
    context_length = cfg.get("context_length", cfg.get("n_positions"))
    hidden_size = cfg.get("hidden_size", cfg.get("n_embd"))
    num_layers = cfg.get("num_layers", cfg.get("n_layer"))
    num_heads = cfg.get("num_heads", cfg.get("n_head"))
    if not all(v is not None for v in (vocab_size, context_length, hidden_size, num_layers, num_heads)):
        raise ValueError(f"Incomplete model configuration: {cfg}")
    gcfg = GPT2Config(
        vocab_size=vocab_size, n_positions=context_length, n_ctx=context_length,
        n_embd=hidden_size, n_layer=num_layers, n_head=num_heads,
        resid_pdrop=bb_dropout, embd_pdrop=bb_dropout, attn_pdrop=bb_dropout
    )
    bb = GPT2Model(gcfg)
    sd = ck.get("model_state_dict", ck)
    if not isinstance(sd, dict):
        raise ValueError("model_state_dict is not a dictionary")
    prefixed = {k[len("transformer."):]: v for k, v in sd.items() if k.startswith("transformer.")}
    if prefixed:
        sd = prefixed
    sd = {k: v for k, v in sd.items() if not k.startswith("lm_head.")}
    missing, unexpected = bb.load_state_dict(sd, strict=False)
    if missing or unexpected:
        raise RuntimeError(
            f"Checkpoint/backbone mismatch. Missing keys (first 8): {missing[:8]}; "
            f"unexpected keys (first 8): {unexpected[:8]}"
        )
    del ck
    if n_layers is not None and n_layers < len(bb.h):
        bb.h = nn.ModuleList(list(bb.h)[:n_layers]); bb.config.n_layer = n_layers
    return bb, cfg["hidden_size"], True, cfg["context_length"], "gpt2"

class SeqClassifier(nn.Module):
    def __init__(self, backbone, hidden, n_labels, causal=True, pos_dim=2, dropout=0.1, family="gpt2"):
        super().__init__()
        self.backbone, self.causal, self.pos_dim, self.family = backbone, causal, pos_dim, family
        self.drop = nn.Dropout(dropout); self.head = nn.Linear(hidden + pos_dim, n_labels); self.prompt = None
    def forward(self, ids, mask, pos=None, return_emb=False):
        B = ids.size(0); lengths = mask.sum(1)
        if self.prompt is not None:                       # prompt tuning: learnable vectors in front of the input
            emb = self.backbone.get_input_embeddings()(ids)
            P = self.prompt.shape[0]
            emb = torch.cat([self.prompt.unsqueeze(0).expand(B, -1, -1).to(emb.dtype), emb], 1)
            mask = torch.cat([torch.ones(B, P, device=ids.device, dtype=mask.dtype), mask], 1); lengths = lengths + P
            h = self.backbone(inputs_embeds=emb, attention_mask=mask, use_cache=False).last_hidden_state
        else:
            h = self.backbone(input_ids=ids, attention_mask=mask, use_cache=False).last_hidden_state
        if self.causal: pooled = h[torch.arange(B, device=h.device), lengths - 1]       # last real token sees the whole input
        else:
            m = mask.unsqueeze(-1).to(h.dtype); pooled = (h * m).sum(1) / m.sum(1)
        feats = pooled if (self.pos_dim == 0 or pos is None) else torch.cat([pooled, pos.to(pooled.dtype)], -1)
        logits = self.head(self.drop(feats))
        return (logits, pooled) if return_emb else logits

def count_params(m):
    tot = sum(p.numel() for p in m.parameters()); tr = sum(p.numel() for p in m.parameters() if p.requires_grad); return tot, tr
def model_size_mb(m):
    return (sum(p.numel() * p.element_size() for p in m.parameters()) + sum(b.numel() * b.element_size() for b in m.buffers())) / 2**20

# ---------------- adapter building blocks ----------------
class LoRADelta(nn.Module):                # additive:  dW x = (alpha/r) B A x   (rsLoRA: alpha/sqrt(r))
    def __init__(self, nx, nf, r, alpha, drop, rs=False):
        super().__init__()
        self.A = nn.Parameter(torch.empty(nx, r).uniform_(-1 / math.sqrt(nx), 1 / math.sqrt(nx))); self.B = nn.Parameter(torch.zeros(r, nf))
        self.scale = alpha / (math.sqrt(r) if rs else r); self.drop = nn.Dropout(drop)
    def forward(self, x): return (self.drop(x) @ self.A) @ self.B * self.scale

class LoRMAPlus(nn.Module):                # multiplicative, additive rank inflation: h' = (I + (alpha/r) B A) h ; B=0 at init
    def __init__(self, d, r, alpha):
        super().__init__()
        self.A = nn.Parameter(torch.empty(r, d).uniform_(-1 / math.sqrt(d), 1 / math.sqrt(d))); self.B = nn.Parameter(torch.zeros(d, r)); self.scale = alpha / r
    def forward(self, h): return h + ((h @ self.A.t()) @ self.B.t()) * self.scale      # = W0x + (alpha/r) B A (W0 x)

class LoRMAPi(nn.Module):                  # multiplicative, permutation-based rank inflation: h' = I_pi(BA) h, I_pi(BA) = I at init
    def __init__(self, d, r):
        super().__init__()
        B = torch.empty(d, r).uniform_(-1 / math.sqrt(d), 1 / math.sqrt(d)); B[:, 0] = 1.0
        A = torch.zeros(r, d); A[0, 0] = 1.0
        self.B, self.A = nn.Parameter(B), nn.Parameter(A)
        i = torch.arange(d); self.register_buffer("idx", (i[None, :] - i[:, None]) % d)   # row i rotated right by i
    def matrix(self): return (self.B @ self.A).gather(1, self.idx)
    def forward(self, h): return h @ self.matrix().t().to(h.dtype)

class QVAdapt(nn.Module):                  # wraps GPT-2's fused c_attn (q|k|v); adapts q and v (k and v for IA3)
    def __init__(self, base, kind, r=8, alpha=16, drop=0.05, rs=False):
        super().__init__()
        nx, nf = conv_dims(base); d = nf // 3; self.base, self.kind, self.d = base, kind, d
        if kind == "lora":         self.q, self.v = LoRADelta(nx, d, r, alpha, drop, rs), LoRADelta(nx, d, r, alpha, drop, rs)
        elif kind == "lorma_plus": self.q, self.v = LoRMAPlus(d, r, alpha), LoRMAPlus(d, r, alpha)
        elif kind == "lorma_pi":   self.q, self.v = LoRMAPi(d, r), LoRMAPi(d, r)
        elif kind == "ia3":        self.lk, self.lv = nn.Parameter(torch.ones(d)), nn.Parameter(torch.ones(d))
        else: raise ValueError(kind)
    def forward(self, x):
        out = self.base(x); d = self.d; q, k, v = out[..., :d], out[..., d:2 * d], out[..., 2 * d:]
        if self.kind == "lora":            q, v = q + self.q(x), v + self.v(x)
        elif self.kind.startswith("lorma"): q, v = self.q(q), self.v(v)
        else:                              k, v = k * self.lk, v * self.lv
        return torch.cat([q, k, v], -1)

class IA3Input(nn.Module):                 # IA3 on the FFN: rescale the activation entering the down-projection
    def __init__(self, base):
        super().__init__(); self.base = base; self.l = nn.Parameter(torch.ones(conv_dims(base)[0]))
    def forward(self, x): return self.base(x * self.l)

class LinAdapt(nn.Module):                 # same adapters for plain nn.Linear projections (non-GPT-2 backbones)
    def __init__(self, base, kind, r=8, alpha=16, drop=0.05, rs=False):
        super().__init__(); self.base, self.kind = base, kind; di, do = base.in_features, base.out_features
        if kind == "lora": self.op = LoRADelta(di, do, r, alpha, drop, rs)
        elif kind == "lorma_plus": self.op = LoRMAPlus(do, r, alpha)
        elif kind == "lorma_pi": self.op = LoRMAPi(do, r)
        elif kind == "ia3": self.l = nn.Parameter(torch.ones(do))
    def forward(self, x):
        y = self.base(x)
        if self.kind == "lora": return y + self.op(x)
        if self.kind == "ia3": return y * self.l
        return self.op(y)

def wrap_linear_targets(root, kind, names, **kw):
    n = 0
    for _, parent in list(root.named_modules()):
        for cname, child in list(parent.named_children()):
            if isinstance(child, nn.Linear) and cname in names: setattr(parent, cname, LinAdapt(child, kind, **kw)); n += 1
    assert n > 0, f"no nn.Linear named {names} found in the backbone"

# ---------------- NF4 quantisation (QLoRA-style frozen base weights, 4-bit packed) ----------------
NF4_CODE = torch.tensor([-1.0, -0.6961928009986877, -0.5250730514526367, -0.39491748809814453, -0.28444138169288635, -0.18477343022823334,
    -0.09105003625154495, 0.0, 0.07958029955625534, 0.16093020141124725, 0.24611230194568634, 0.33791524171829224,
    0.44070982933044434, 0.5626170039176941, 0.7229568362236023, 1.0])
def nf4_quantize(w, block=64):
    flat = w.detach().float().reshape(-1); n = flat.numel(); pad = (-n) % block
    if pad: flat = F.pad(flat, (0, pad))
    blocks = flat.view(-1, block); absmax = blocks.abs().amax(dim=1).clamp_min(1e-8)
    normed = (blocks / absmax[:, None]).reshape(-1)
    code = NF4_CODE.to(flat.device); mids = (code[:-1] + code[1:]) / 2
    idx = torch.bucketize(normed, mids).to(torch.uint8)
    return ((idx[0::2] << 4) | idx[1::2]), absmax.half(), n
def nf4_dequant(packed, absmax, n, shape, block=64):
    code = NF4_CODE.to(packed.device)
    idx = torch.stack([(packed >> 4).long(), (packed & 0x0F).long()], dim=1).reshape(-1)
    return (code[idx].view(-1, block) * absmax.float()[:, None]).reshape(-1)[:n].view(shape)

class NF4Conv1D(nn.Module):
    def __init__(self, conv, block=64):
        super().__init__(); nx, nf = conv.weight.shape; self.nx, self.nf, self.block = nx, nf, block
        packed, absmax, n = nf4_quantize(conv.weight.data, block); self.n = n
        self.register_buffer("packed", packed); self.register_buffer("absmax", absmax); self.register_buffer("bias", conv.bias.data.clone())
    def forward(self, x):
        W = nf4_dequant(self.packed, self.absmax, self.n, (self.nx, self.nf), self.block).to(x.dtype)
        return x @ W + self.bias.to(x.dtype)

def conv_dims(m):
    if isinstance(m, NF4Conv1D): return m.nx, m.nf
    return tuple(m.weight.shape)                     # HF Conv1D weight is [in, out]

METHODS = ("full", "lora", "rslora", "qlora", "lorma_plus", "lorma_pi", "ia3", "prompt")
def apply_method(model, method, hp):
    assert method in METHODS, method
    if method == "full":
        for p in model.parameters(): p.requires_grad = True
        return model
    for p in model.backbone.parameters(): p.requires_grad = False
    r, alpha, drop = hp.get("r", 8), hp.get("alpha", 16), hp.get("adapter_dropout", 0.05)
    if method == "prompt":
        emb = model.backbone.get_input_embeddings().weight; P = hp.get("n_prompt", 20)
        init = emb[torch.randint(0, emb.shape[0], (P,))].detach().clone()
        model.prompt = nn.Parameter(init); return model
    kind = {"lora": "lora", "rslora": "lora", "qlora": "lora", "lorma_plus": "lorma_plus", "lorma_pi": "lorma_pi", "ia3": "ia3"}[method]
    kw = dict(r=r, alpha=alpha, drop=drop, rs=(method == "rslora"))
    if model.family == "gpt2":
        for blk in model.backbone.h:
            if method == "qlora":
                for owner, name in ((blk.attn, "c_attn"), (blk.attn, "c_proj"), (blk.mlp, "c_fc"), (blk.mlp, "c_proj")):
                    setattr(owner, name, NF4Conv1D(getattr(owner, name)))
            assert isinstance(blk.attn.c_attn, (Conv1D, NF4Conv1D)), "unexpected c_attn type in this transformers version"
            blk.attn.c_attn = QVAdapt(blk.attn.c_attn, kind, **kw)
            if kind == "ia3": blk.mlp.c_proj = IA3Input(blk.mlp.c_proj)
    else:
        if method == "qlora": raise NotImplementedError("QLoRA is implemented for the GPT-2 family only")
        names = ("k_proj", "v_proj") if kind == "ia3" else ("q_proj", "v_proj")
        wrap_linear_targets(model.backbone, kind, names, **({} if kind == "ia3" else kw))
    return model


In [ ]:
# ================= CELL 8 — training / evaluation harness + RESUMABLE CHECKPOINTS =================
# Recovery strategy:
#   latest.pt: full model + optimizer/scheduler/scaler + epoch + RNG state (resume after interruption)
#   best.pt: best validation model weights (use for inference)
#   result.json/logits/embeddings: completed experiment artifacts
K = len(LABELS)
HP_DEFAULT = {
    "full": dict(lr=3e-5), "lora": dict(lr=5e-4, r=8, alpha=16), "rslora": dict(lr=5e-4, r=8, alpha=16), "qlora": dict(lr=5e-4, r=8, alpha=16),
    "lorma_plus": dict(lr=5e-4, r=8, alpha=16), "lorma_pi": dict(lr=5e-4, r=8), "ia3": dict(lr=3e-3), "prompt": dict(lr=1e-2, n_prompt=20),
}
CFG_DEFAULT = dict(arch=None, method="full", ctx=True, epochs=8, patience=3, bs=16, warmup=0.1, wd=0.01, head_lr=1e-3, label_smooth=0.05,
                   class_weights=True, bb_dropout=0.1, k_left=3, k_right=1, seed=SEED, n_layers=None, kd=None, merge_dev=False,
                   fixed_epochs=None, hp={}, force=False, resume=True)
use_amp = device.type == "cuda"

def atomic_torch_save(obj, path):
    """Write checkpoint to a temporary file then atomically replace destination."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)

def cpu_state_dict(model):
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

def checkpoint_rng_state():
    state = {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch": torch.get_rng_state(),
    }
    if torch.cuda.is_available():
        state["cuda"] = torch.cuda.get_rng_state_all()
    return state

def restore_rng_state(state):
    if not state:
        return
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"])
    if torch.cuda.is_available() and state.get("cuda") is not None:
        torch.cuda.set_rng_state_all(state["cuda"])

@torch.no_grad()
def predict(model, exs, bs=64, emb=False):
    model.eval(); lg = np.zeros((len(exs), K), np.float32); em = None
    order = sorted(range(len(exs)), key=lambda j: len(exs[j]["ids"]))
    for s in range(0, len(order), bs):
        bidx = order[s:s + bs]; ids, mask, pos, _ = collate(exs, bidx)
        with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
            out = model(ids.to(device), mask.to(device), pos.to(device), return_emb=emb)
        if emb:
            out, pooled = out
            if em is None: em = np.zeros((len(exs), pooled.shape[-1]), np.float16)
            em[bidx] = pooled.float().cpu().numpy().astype(np.float16)
        lg[bidx] = out.float().cpu().numpy()
    return (lg, em) if emb else lg

def snapshot(model):
    return {n: p.detach().to("cpu", copy=True) for n, p in model.named_parameters() if p.requires_grad}

def restore(model, snap):
    with torch.no_grad():
        for n, p in model.named_parameters():
            if n in snap: p.copy_(snap[n].to(p.device))

def run_one(name, dom, cfg):
    c = {**CFG_DEFAULT, **cfg}
    if not ARCHS:
        raise RuntimeError("ARCHS is empty; configure the Phase 1 checkpoint in Cell 6.")
    c["arch"] = c["arch"] or next(iter(ARCHS))
    hp = {**HP_DEFAULT[c["method"]], **c["hp"]}
    rdir = WORK / "runs" / f"{name}_{dom}"
    rdir.mkdir(parents=True, exist_ok=True)
    latest_path, best_path = rdir / "latest.pt", rdir / "best.pt"
    if (rdir / "result.json").exists() and not c["force"]:
        print(f"[{name}/{dom}] already done - skipping (force=True to redo)")
        return json.load(open(rdir / "result.json", encoding="utf-8"))

    seed_all(c["seed"]); t_all = time.time()
    tr_docs = DOCS[dom]["train"] + (DOCS[dom]["dev"] if c["merge_dev"] else [])
    kw = dict(ctx=c["ctx"], k_left=c["k_left"], k_right=c["k_right"])
    ex_tr = build_examples(tr_docs, **kw)
    ex_dv = build_examples(DOCS[dom]["dev"], **kw)
    ex_te = build_examples(DOCS[dom]["test"], **kw)
    y_tr = np.array([e["y"] for e in ex_tr]); y_dv = np.array([e["y"] for e in ex_dv])

    bb, hidden, causal, npos, fam = build_backbone(c["arch"], c["bb_dropout"], c["n_layers"])
    model = SeqClassifier(bb, hidden, K, causal, 2, 0.1, fam)
    apply_method(model, c["method"], hp); model.to(device)
    longest = max(len(e["ids"]) for e in ex_tr + ex_dv + ex_te) + (hp.get("n_prompt", 0) if c["method"] == "prompt" else 0)
    assert longest <= npos, f"input length {longest} exceeds the model context {npos}"

    tot_p, tr_p = count_params(model); size_mb = model_size_mb(model)
    cnt = np.bincount(y_tr, minlength=K).astype(float); cw = None
    if c["class_weights"]:
        w = 1 / np.sqrt(np.maximum(cnt, 1)); w = w * cnt.sum() / (w * cnt).sum()
        cw = torch.tensor(w, dtype=torch.float32, device=device)
    T_logits = None
    if c["kd"]:
        assert not c["merge_dev"], "KD + merge_dev not supported"
        T_logits = np.load(WORK / "runs" / f"teacher_{dom}.npy")
        assert len(T_logits) == len(ex_tr), "teacher logits do not match the training examples"

    other = [p for n, p in model.named_parameters() if p.requires_grad and not n.startswith("head.")]
    opt = torch.optim.AdamW(
        [{"params": other, "lr": hp["lr"]},
         {"params": list(model.head.parameters()), "lr": c["head_lr"]}],
        weight_decay=c["wd"])
    epochs = c["fixed_epochs"] or c["epochs"]
    spe = math.ceil(len(ex_tr) / c["bs"]); total = spe * epochs
    warm = int(c["warmup"] * total)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda step: min((step + 1) / max(1, warm),
                              max(0.0, (total - step) / max(1, total - warm))))
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    rng = random.Random(c["seed"])
    trainable = [p for p in model.parameters() if p.requires_grad]

    start_epoch, best, best_ep, bad, hist = 1, -1.0, 0, 0, []
    best_state = None
    if c["resume"] and latest_path.exists() and not c["force"]:
        try:
            ck = torch.load(latest_path, map_location="cpu", weights_only=False)
            model.load_state_dict(ck["model_state"])
            opt.load_state_dict(ck["optimizer_state"])
            sched.load_state_dict(ck["scheduler_state"])
            scaler.load_state_dict(ck["scaler_state"])
            start_epoch = int(ck["epoch"]) + 1
            best, best_ep, bad = float(ck["best_f1"]), int(ck["best_epoch"]), int(ck["bad_epochs"])
            hist = ck.get("history", [])
            # Backward-compatible: older checkpoints embedded best weights.
            # New checkpoints keep best weights only in best.pt to avoid duplication.
            best_state = ck.get("best_model_state")
            if best_state is None and best_path.exists():
                try:
                    best_ck = torch.load(best_path, map_location="cpu", weights_only=False)
                    best_state = best_ck.get("model_state")
                    del best_ck
                except Exception as best_e:
                    print("   Could not load separate best.pt; continuing with latest weights:", repr(best_e))
            restore_rng_state(ck.get("rng_state"))
            print(f"[{name}/{dom}] RESUMING from epoch {start_epoch} (latest checkpoint epoch {ck['epoch']})")
        except Exception as e:
            print(f"[{name}/{dom}] WARNING: could not resume checkpoint ({e}); starting fresh.")
            start_epoch, best, best_ep, bad, hist, best_state = 1, -1.0, 0, 0, [], None
    if c["force"]:
        print(f"[{name}/{dom}] force=True: starting a fresh run; old checkpoints will be replaced.")

    if device.type == "cuda": torch.cuda.reset_peak_memory_stats()
    print(f"[{name}/{dom}] arch={c['arch']} method={c['method']} ctx={c['ctx']} | train {len(ex_tr)} sents | "
          f"params total {tot_p/1e6:.1f}M trainable {tr_p/1e6:.3f}M ({100*tr_p/tot_p:.2f}%) | size {size_mb:.0f} MB")
    t0 = time.time()
    for ep in range(start_epoch, epochs + 1):
        model.train(); tl, nb = 0.0, 0
        for bidx in make_batches(ex_tr, c["bs"], True, rng):
            ids, mask, pos, y = collate(ex_tr, bidx)
            ids, mask, pos, y = ids.to(device), mask.to(device), pos.to(device), y.to(device)
            with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
                logits = model(ids, mask, pos)
            loss = F.cross_entropy(logits.float(), y, weight=cw, label_smoothing=c["label_smooth"])
            if T_logits is not None:
                temp, alpha = c["kd"].get("T", 2.0), c["kd"].get("alpha", 0.5)
                tl_ = torch.tensor(T_logits[bidx], device=device)
                kl = F.kl_div(F.log_softmax(logits.float() / temp, -1),
                              F.softmax(tl_ / temp, -1), reduction="batchmean") * temp * temp
                loss = alpha * loss + (1 - alpha) * kl
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(trainable, 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            tl += loss.item(); nb += 1

        if c["merge_dev"]:
            epoch_f1 = None
            print(f"   epoch {ep}/{epochs} loss {tl/max(nb,1):.4f} (train+dev refit: no validation)")
            best_state = cpu_state_dict(model); best_ep = ep
        else:
            rep_ep = prf_report(y_dv, predict(model, ex_dv).argmax(1), K)
            epoch_f1 = float(rep_ep["macro_f1"])
            hist.append(dict(epoch=ep, loss=tl / max(nb, 1), dev_macro_f1=epoch_f1))
            print(f"   epoch {ep}/{epochs} loss {tl/max(nb,1):.4f} | dev macro-F1 {epoch_f1:.4f} "
                  f"(P {rep_ep['macro_p']:.3f} R {rep_ep['macro_r']:.3f}) acc {rep_ep['acc']:.3f}")
            if epoch_f1 > best + 1e-4:
                best, best_ep, bad = epoch_f1, ep, 0
                best_state = cpu_state_dict(model)
                atomic_torch_save({
                    "model_state": best_state, "epoch": ep, "best_f1": best,
                    "best_epoch": best_ep, "arch": c["arch"], "method": c["method"],
                    "domain": dom, "experiment": name, "config": c,
                }, best_path)
            else:
                bad += 1

        # Save after EVERY completed epoch. A runtime interruption loses at most the in-progress epoch.
        atomic_torch_save({
            "format_version": 1, "experiment": name, "domain": dom, "epoch": ep,
            "model_state": cpu_state_dict(model), "optimizer_state": opt.state_dict(),
            "scheduler_state": sched.state_dict(), "scaler_state": scaler.state_dict(),
            "best_f1": best, "best_epoch": best_ep, "bad_epochs": bad,
            # Do not embed best_model_state here: best.pt is the single rolling copy.
            "history": hist, "rng_state": checkpoint_rng_state(),
            "config": c, "hp": hp, "saved_unix": time.time(),
        }, latest_path)
        print(f"   checkpoint saved: {latest_path.name} (epoch {ep})")

        # Remote per-epoch publishing is intentionally disabled by default.
        # This prevents creating a new Kaggle Dataset version every few epochs.
        # Local rolling latest.pt/best.pt remain available for same-session resume.


        if not c["merge_dev"] and bad >= c["patience"] and not c["fixed_epochs"]:
            print("   early stop"); break

    # If all epochs were already completed in a valid checkpoint, use its best state.
    if best_state is not None:
        model.load_state_dict(best_state)
        atomic_torch_save({
            "model_state": cpu_state_dict(model), "epoch": best_ep or epochs,
            "best_f1": best, "best_epoch": best_ep, "arch": c["arch"],
            "method": c["method"], "domain": dom, "experiment": name,
            "config": c, "saved_unix": time.time(),
        }, best_path)

    train_sec = time.time() - t0
    peak_mb = torch.cuda.max_memory_allocated() / 2**20 if device.type == "cuda" else 0.0
    t1 = time.time(); lg_dv, em_dv = predict(model, ex_dv, emb=True)
    sps = len(ex_dv) / max(time.time() - t1, 1e-6)
    lg_te, em_te = predict(model, ex_te, emb=True)
    lg_tr, em_tr = predict(model, ex_tr, emb=True)
    for nm, arr in dict(dev_logits=lg_dv, test_logits=lg_te, train_logits=lg_tr,
                        dev_emb=em_dv, test_emb=em_te, train_emb=em_tr).items():
        np.save(rdir / f"{nm}.npy", arr)
    np.save(rdir / "train_y.npy", y_tr)
    rep = prf_report(y_dv, lg_dv.argmax(1), K)
    res = dict(name=name, domain=dom, arch=c["arch"], method=c["method"], ctx=c["ctx"],
               kd=bool(c["kd"]), merge_dev=c["merge_dev"], n_layers=c["n_layers"],
               best_epoch=best_ep or epochs, epochs_run=len(hist) or epochs,
               dev_macro_f1=rep["macro_f1"], dev_macro_p=rep["macro_p"], dev_macro_r=rep["macro_r"],
               dev_macro_f1_all13=rep["macro_f1_all"], dev_acc=rep["acc"],
               params_total=tot_p, params_trainable=tr_p, trainable_pct=100 * tr_p / tot_p,
               model_size_mb=size_mb, peak_gpu_mb=peak_mb, train_sec=train_sec,
               sents_per_sec=sps, cfg=json.dumps({**c, "hp": hp}, default=str),
               **{f"f1_{l}": float(rep["f1"][i]) for i, l in enumerate(LABELS)})
    tmp_result = rdir / "result.json.tmp"
    with open(tmp_result, "w", encoding="utf-8") as f:
        json.dump({**res, "confusion": rep["cm"].tolist(), "history": hist}, f, indent=2)
    os.replace(tmp_result, rdir / "result.json")
    newrow = pd.DataFrame([res])
    if LEDGER.exists():
        old = pd.read_csv(LEDGER)
        old = old[~((old["name"] == name) & (old["domain"] == dom))]
        newrow = pd.concat([old, newrow], ignore_index=True)
    newrow.to_csv(LEDGER, index=False)
    if PUBLISH_AT_EACH_EXPERIMENT_END and DURABLE_PUBLISH_ENABLED:
        publish_recovery_dataset(reason=f"{name}_{dom}_experiment_complete")
    print(f"[{name}/{dom}] DONE dev macro-F1 {rep['macro_f1']:.4f} @epoch {res['best_epoch']} | "
          f"{train_sec/60:.1f} min | peak {peak_mb:.0f} MB | {sps:.0f} sent/s")
    del model, opt, scaler, best_state
    gc.collect(); torch.cuda.empty_cache()
    return res

def run_experiment(name, domains=("CL", "IT"), **cfg):
    return {d: run_one(name, d, cfg) for d in domains}

def make_teacher(run_names, dom):
    """Geometric-mean ensemble of finished runs' train logits -> teacher for KD."""
    lp = [np.log(softmax_np(np.load(WORK / "runs" / f"{n}_{dom}" / "train_logits.npy")) + 1e-9)
          for n in run_names]
    np.save(WORK / "runs" / f"teacher_{dom}.npy", np.mean(lp, 0).astype(np.float32))
    print(f"teacher_{dom}: ensemble of {run_names}")

def show_ledger(cols=None):
    if not LEDGER.exists():
        print("No completed experiments in ledger yet.")
        return pd.DataFrame()
    df = pd.read_csv(LEDGER).drop_duplicates(subset=["name", "domain"], keep="last")
    cols = cols or ["name", "domain", "arch", "method", "ctx", "kd", "dev_macro_f1",
                    "dev_macro_p", "dev_macro_r", "best_epoch", "params_trainable",
                    "model_size_mb", "peak_gpu_mb", "train_sec"]
    print(df[cols].round(4).to_string(index=False))
    return df


In [ ]:
# ================= CELL 9 — smoke tests (run once BEFORE any long training; ~2-3 min) =================
def smoke_tests(arch=None):
    if not ARCHS:
        raise RuntimeError(
            "ARCHS is empty: no compatible Phase 1 checkpoint was detected. "
            "Run CELL 2 to inspect checkpoints, then edit ARCHS in CELL 6 "
            "with the correct checkpoint path and family='gpt2' for the GPT-2 checkpoint."
        )
    arch = arch or next(iter(ARCHS))
    if arch not in ARCHS:
        raise KeyError(f"Unknown architecture {arch!r}. Registered: {list(ARCHS)}")
    print("testing architecture:", arch)
    bb, hidden, causal, npos, fam = build_backbone(arch, bb_dropout=0.0)
    base = SeqClassifier(bb, hidden, K, causal, 2, 0.0, fam).eval()
    exs = build_examples(DOCS["CL"]["train"][:1], True)[:4]; ids, mask, pos, y = collate(exs, list(range(len(exs))))
    with torch.no_grad(): ref = base(ids, mask, pos)
    rows = []
    for m in ("lora", "rslora", "lorma_plus", "lorma_pi", "ia3", "qlora", "prompt"):
        mdl = copy.deepcopy(base); apply_method(mdl, m, {"n_prompt": 8}); mdl.eval()
        with torch.no_grad(): out = mdl(ids, mask, pos)
        diff = (out - ref).abs().max().item(); tot, tr = count_params(mdl)
        if m in ("qlora", "prompt"): ok = bool(torch.isfinite(out).all()); note = "changes outputs by design (quantisation / extra tokens)"
        else: ok = diff < 1e-3; note = "must equal the base model at initialisation"
        rows.append(dict(method=m, max_logit_diff=round(diff, 5), trainable_M=round(tr / 1e6, 3), size_MB=round(model_size_mb(mdl)), ok=ok, check=note))
        del mdl; gc.collect()
    pi = apply_method(copy.deepcopy(base), "lorma_pi", {}).backbone.h[0].attn.c_attn.q.matrix()
    rows.append(dict(method="lorma_pi: matrix==I", max_logit_diff=round((pi - torch.eye(pi.shape[0])).abs().max().item(), 6), ok=bool(torch.allclose(pi, torch.eye(pi.shape[0]), atol=1e-6)), check="permutation inflation starts as identity"))
    df = pd.DataFrame(rows); print(df.to_string(index=False)); assert df["ok"].all(), "an adapter test failed - do not run that method"
    if device.type == "cuda":
        for m in ("lorma_plus", "full"):
            mdl = SeqClassifier(*build_backbone(arch, 0.0)[:2], K, causal, 2, 0.0, fam); apply_method(mdl, m, {}); mdl.to(device).train()
            ids_, mask_, pos_, y_ = [t.to(device) for t in collate(exs, list(range(len(exs))))]
            opt = torch.optim.AdamW([p for p in mdl.parameters() if p.requires_grad], lr=1e-3 if m != "full" else 1e-4); L = []
            for _ in range(25):
                loss = F.cross_entropy(mdl(ids_, mask_, pos_), y_); opt.zero_grad(); loss.backward(); opt.step(); L.append(loss.item())
            print(f"overfit test [{m}]: loss {L[0]:.3f} -> {L[-1]:.3f}"); assert L[-1] < 0.7 * L[0], f"{m} did not learn"
            del mdl, opt; gc.collect(); torch.cuda.empty_cache()
    print("\nALL SMOKE TESTS PASSED")
smoke_tests()


### Smoke tests passed? Then run the experiments in this order (compulsory ones first). Each run skips itself if its result already exists, so it is safe to re-run after a crash.

In [ ]:
# ================= CELL 10 — EXPERIMENT PLAN =================
# Only the GPT-2-style 3,000-step checkpoint is currently registered.
# This does NOT satisfy four-architecture coverage until the other Phase 1 models
# are connected and independently smoke-tested.
if not ARCHS:
    raise RuntimeError("ARCHS is empty. Fix checkpoint registration before experiments.")
ARCH_DEFAULT = "gpt2_a"
PLAN = {
    "E0b_full_ctx":  dict(method="full",       ctx=True,  arch=ARCH_DEFAULT),
    "E2_lorma_plus": dict(method="lorma_plus", ctx=True,  arch=ARCH_DEFAULT),
    "E1_lora":       dict(method="lora",       ctx=True,  arch=ARCH_DEFAULT),
    "E3_ia3":        dict(method="ia3",        ctx=True,  arch=ARCH_DEFAULT),
    "E4_qlora":      dict(method="qlora",      ctx=True,  arch=ARCH_DEFAULT),
    "E5_prompt":     dict(method="prompt",     ctx=True,  arch=ARCH_DEFAULT),
    "E0a_full_sent": dict(method="full",       ctx=False, arch=ARCH_DEFAULT),
}
print(pd.DataFrame(PLAN).T.to_string())


In [ ]:
# ================= CELL 11 — RUN A: baseline + the compulsory LoRMA experiment (both domains) =================
for nm in ("E0b_full_ctx", "E2_lorma_plus"):
    run_experiment(nm, **PLAN[nm])
show_ledger();


In [ ]:
# ================= CELL 12 — RUN B: the other adaptation techniques + context ablation =================
for nm in ("E1_lora", "E3_ia3", "E4_qlora", "E0a_full_sent"):     # add "E5_prompt" if you want a sixth technique
    run_experiment(nm, **PLAN[nm])
show_ledger();


In [ ]:
from pathlib import Path
import json
import shutil

ROOT = Path("/kaggle/working/phase3_rr/runs")

free = shutil.disk_usage("/kaggle/working").free
print(f"Free disk: {free / 2**30:.2f} GiB")

for name in [
    "E0a_full_sent_CL", "E0a_full_sent_IT",
    "E1_lora_CL", "E1_lora_IT",
    "E3_ia3_CL", "E3_ia3_IT",
    "E4_qlora_CL", "E4_qlora_IT",
]:
    d = ROOT / name
    print(f"\n{name}:")
    if not d.exists():
        print("  Run directory missing")
        continue

    for fname in ["best.pt", "latest.pt", "result.json"]:
        p = d / fname
        if p.exists():
            print(f"  {fname}: {p.stat().st_size / 2**30:.2f} GiB")

    rp = d / "result.json"
    if rp.exists():
        try:
            data = json.loads(rp.read_text())
            print("  result.json:", data)
        except Exception as e:
            print("  Could not read result.json:", e)

In [ ]:
# ================= CELL 13 — RUN C: knowledge distillation (teacher = ensemble of finished runs, student = half-depth model from Phase 1 weights) =================
TEACHERS = ["E0b_full_ctx", "E2_lorma_plus"]          # choose the strongest finished runs
for dom in ("CL", "IT"): make_teacher(TEACHERS, dom)
# Student initialised from the same Phase 1 weights but keeping only the first 6 layers (set n_layers=None and change `arch` to use a smaller Phase 1 model instead).
run_experiment("E6_kd_student", method="full", ctx=True, arch=ARCH_DEFAULT, n_layers=6, kd=dict(T=2.0, alpha=0.5))
# Control without distillation (same student, hard labels only) so the benefit of KD is measurable:
run_experiment("E6_student_nokd", method="full", ctx=True, arch=ARCH_DEFAULT, n_layers=6)
show_ledger();


In [ ]:
# ================= CELL 14 — comparison tables, per-class F1, confusion matrices, architecture coverage =================
import matplotlib; import matplotlib.pyplot as plt
df = pd.read_csv(LEDGER).drop_duplicates(subset=["name", "domain"], keep="last")
df = df[~df["merge_dev"].astype(bool)]
main = df.pivot_table(index="name", columns="domain", values="dev_macro_f1").round(4); main["mean"] = main.mean(1)
print("dev macro-F1 by experiment and domain\n", main.sort_values("mean", ascending=False).to_string())
eff = df.groupby("name")[["params_trainable", "model_size_mb", "peak_gpu_mb", "train_sec", "sents_per_sec"]].mean().round(1); print("\nefficiency (mean over domains)\n", eff.to_string())
pc = df.set_index(["name", "domain"])[[f"f1_{l}" for l in LABELS]].round(3); pc.columns = LABELS; print("\nper-class F1\n", pc.to_string())
print("\narchitecture coverage (all four Phase 1 architectures must appear):\n", df.groupby("arch")["name"].nunique().to_string())
def plot_cm(name, dom):
    r = json.load(open(WORK / "runs" / f"{name}_{dom}" / "result.json")); cm = np.array(r["confusion"]); cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1)
    fig, ax = plt.subplots(figsize=(8, 7)); im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(K)); ax.set_xticklabels(LABELS, rotation=90, fontsize=7); ax.set_yticks(range(K)); ax.set_yticklabels(LABELS, fontsize=7)
    ax.set_title(f"{name} / {dom}  (row-normalised)"); plt.colorbar(im); plt.tight_layout(); plt.savefig(WORK / f"cm_{name}_{dom}.png", dpi=120); plt.show()
BEST = main["mean"].idxmax(); print("\nbest by mean dev macro-F1:", BEST)
for dom in ("CL", "IT"): plot_cm(BEST, dom)


In [ ]:
# ================= CELL 15 — post-hoc improvements with NO retraining: ensembling, transition smoothing, kNN interpolation =================
def load_arr(name, dom, nm): return np.load(WORK / "runs" / f"{name}_{dom}" / f"{nm}.npy")
def ens_probs(names, dom, split):
    lp = [np.log(softmax_np(load_arr(n, dom, f"{split}_logits")) + 1e-9) for n in names]; return softmax_np(np.mean(lp, 0))
def posthoc_probs(names, dom, split, smooth=0.0, knn_lam=1.0, k=10, tau=0.1):
    p = ens_probs(names, dom, split)
    if knn_lam < 1.0:   # neighbours from train embeddings of the FIRST run in `names`
        p = knn_interpolate(load_arr(names[0], dom, f"{split}_emb").astype(np.float32), load_arr(names[0], dom, "train_emb").astype(np.float32),
                            load_arr(names[0], dom, "train_y"), p, K, k=k, tau=tau, lam=knn_lam)
    return p
def predict_labels(probs, exs, logT, logS, smooth): return smooth_split(probs, exs, logT, logS, smooth) if smooth > 0 else probs.argmax(1)

def tune_posthoc(names, dom):
    ex_dv = build_examples(DOCS[dom]["dev"], True); y = np.array([e["y"] for e in ex_dv])
    logT, logS = transition_logprobs(DOCS[dom]["train"], LABEL2ID, K); rows = []
    for knn_lam in (1.0, 0.8, 0.6):
        p = posthoc_probs(names, dom, "dev", knn_lam=knn_lam)
        for sm in (0.0, 0.25, 0.5, 1.0):
            rows.append(dict(domain=dom, knn_lam=knn_lam, smooth=sm, dev_macro_f1=prf_report(y, predict_labels(p, ex_dv, logT, logS, sm), K)["macro_f1"]))
    t = pd.DataFrame(rows); base = t[(t.knn_lam == 1.0) & (t.smooth == 0.0)].dev_macro_f1.iloc[0]
    print(t.round(4).to_string(index=False)); print(f"baseline {base:.4f}; best {t.dev_macro_f1.max():.4f}"); return t, base

CANDIDATES = {dom: [BEST] for dom in ("CL", "IT")}      # list one or more finished run names per domain (an ensemble if several)
for dom in ("CL", "IT"): _ = tune_posthoc(CANDIDATES[dom], dom)
print("\nRule: adopt a post-hoc setting only if it gains >= 0.01 on BOTH domains; dev has very few documents and small gains are noise.")


In [ ]:

# ================= CELL 16 — FINAL PREDICTIONS + VALIDATION + ZIP =================

from pathlib import Path
from collections import defaultdict, Counter
import json
import zipfile
import time

# Ensure output directory exists.
WORK = Path(WORK)
WORK.mkdir(parents=True, exist_ok=True)

# Choose the validated best configuration.
FINAL = {
    "CL": dict(names=[BEST], knn_lam=1.0, smooth=0.0),
    "IT": dict(names=[BEST], knn_lam=1.0, smooth=0.0),
}

PREDICTIONS = {}

# ---------- Generate and validate per-domain predictions ----------
for dom in ("CL", "IT"):
    f = FINAL[dom]

    ex_te = build_examples(
        DOCS[dom]["test"],
        ctx=PLAN[f["names"][0]]["ctx"],
    )

    probs = posthoc_probs(
        f["names"],
        dom,
        "test",
        knn_lam=f["knn_lam"],
    )

    logT, logS = transition_logprobs(
        DOCS[dom]["train"] + DOCS[dom]["dev"],
        LABEL2ID,
        K,
    )

    yhat = predict_labels(probs, ex_te, logT, logS, f["smooth"])

    assert len(yhat) == len(ex_te), (
        f"{dom}: got {len(yhat)} predictions for {len(ex_te)} examples"
    )

    by_doc = defaultdict(list)
    ordered = sorted(
        zip(ex_te, yhat),
        key=lambda pair: (pair[0]["doc"], pair[0]["i"]),
    )

    for e, y_ in ordered:
        by_doc[e["doc"]].append(LABELS[int(y_)])

    pred_obj = {}

    for d in DOCS[dom]["test"]:
        doc_id = d["id"]
        labs = by_doc[doc_id]

        assert len(labs) == len(d["sents"]), (
            f"{dom}/{doc_id}: predicted {len(labs)} labels "
            f"for {len(d['sents'])} sentences"
        )
        assert all(lab in LABEL2ID for lab in labs), (
            f"{dom}/{doc_id}: invalid label found"
        )

        pred_obj[doc_id] = labs

    # Validate document IDs and order against the source test object.
    assert list(pred_obj.keys()) == list(TEST_OBJ[dom].keys()), (
        f"{dom}: document IDs/order mismatch"
    )

    for doc_id, sentences in TEST_OBJ[dom].items():
        assert len(pred_obj[doc_id]) == len(sentences), (
            f"{dom}/{doc_id}: sentence count mismatch"
        )

    PREDICTIONS[dom] = pred_obj

    out_file = WORK / f"prediction_{dom}.json"
    with open(out_file, "w", encoding="utf-8") as f_out:
        json.dump(pred_obj, f_out, ensure_ascii=False, indent=2)

    print(f"{dom}: {sum(map(len, pred_obj.values()))} sentence labels")
    print(
        "  label distribution:",
        dict(Counter(l for labs in pred_obj.values() for l in labs)),
    )
    print("  saved:", out_file)


# ---------- Merge into required flat document-ID mapping ----------
all_ids = list(PREDICTIONS["CL"].keys()) + list(PREDICTIONS["IT"].keys())
assert len(all_ids) == len(set(all_ids)), (
    "Document IDs overlap between CL and IT; refusing to overwrite predictions."
)

combined = {
    **PREDICTIONS["CL"],
    **PREDICTIONS["IT"],
}

combined_file = WORK / "prediction.json"
with open(combined_file, "w", encoding="utf-8") as f_out:
    json.dump(combined, f_out, ensure_ascii=False, indent=2)

# ---------- Validate the flat JSON ----------
assert list(combined.keys()) == (
    list(TEST_OBJ["CL"].keys()) + list(TEST_OBJ["IT"].keys())
), "Combined document IDs/order do not match the expected CL + IT order."

for dom in ("CL", "IT"):
    for doc_id, sentences in TEST_OBJ[dom].items():
        assert doc_id in combined, f"Missing document: {doc_id}"
        assert len(combined[doc_id]) == len(sentences), (
            f"{doc_id}: label count differs from sentence count"
        )
        assert all(label in LABEL2ID for label in combined[doc_id]), (
            f"{doc_id}: invalid rhetorical-role label"
        )

print("\nCombined prediction file:", combined_file)
print("Total documents:", len(combined))
print("Total sentence labels:", sum(len(v) for v in combined.values()))


# ---------- Create submission ZIP ----------
# Keep prediction.json directly at the ZIP root.
# Per-domain debug JSON files are deliberately excluded from submission.zip.
zip_path = Path("/kaggle/working/submission.zip")

with zipfile.ZipFile(
    zip_path, "w", compression=zipfile.ZIP_DEFLATED
) as zf:
    zf.write(combined_file, "prediction.json")

with zipfile.ZipFile(zip_path, "r") as zf:
    assert zf.namelist() == ["prediction.json"], (
        f"Unexpected submission ZIP contents: {zf.namelist()}"
    )
    check = json.load(zf.open("prediction.json"))

    assert list(check.keys()) == list(combined.keys())
    assert check == combined

print("Submission ZIP validated:", zip_path)
print("ZIP size (MB):", round(zip_path.stat().st_size / (1024**2), 2))


# ================= DURABLE RECOVERY BUNDLE =================
RECOVERY_DIR = Path("/kaggle/working/phase3_recovery")
RECOVERY_DIR.mkdir(parents=True, exist_ok=True)

RECOVERY_ZIP = Path("/kaggle/working/phase3_recovery.zip")

manifest = {
    "created_unix": time.time(),
    "project": "CS781 Phase 3 Rhetorical Role Prediction",
    "source_dir": str(WORK),
    "checkpoint_policy": (
        "Rolling latest.pt stores resumable state; best.pt stores best validation weights. "
        "No epoch-numbered checkpoints are retained; logits/embeddings are excluded from recovery ZIP."
    ),
    "contains": [
         "Rolling latest.pt and best.pt checkpoints",
        "Small result.json summaries and ledger",
        "Prediction JSON files, when present (no logits/embeddings)",
        "Recovery manifest",
    ],
    "durability_note": (
        "This ZIP remains session-local until downloaded or published "
        "as a version of a Kaggle Dataset."
    ),
}

manifest_file = RECOVERY_DIR / "recovery_manifest.json"
with open(manifest_file, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

if RECOVERY_ZIP.exists():
    RECOVERY_ZIP.unlink()

with zipfile.ZipFile(
    RECOVERY_ZIP, "w", compression=zipfile.ZIP_DEFLATED
) as zf:
    # Include only rolling checkpoints and small recovery metadata.
    # Large logits/embedding arrays (*.npy) are intentionally excluded.
    allowed_root_files = {
        "ledger.csv", "prediction.json", "prediction_CL.json", "prediction_IT.json"
    }
    allowed_run_files = {"latest.pt", "best.pt", "result.json"}
    for src in WORK.rglob("*"):
        if not src.is_file() or src.name.endswith(".tmp"):
            continue
        if src.resolve() == RECOVERY_ZIP.resolve():
            continue
        if RECOVERY_DIR in src.parents:
            continue

        rel = src.relative_to(WORK)
        if rel.parts and rel.parts[0] == "runs":
            if src.name not in allowed_run_files:
                continue
        elif rel.parts and len(rel.parts) == 1:
            if src.name not in allowed_root_files:
                continue
        else:
            continue

        zf.write(src, Path("phase3_rr") / rel)

    zf.write(manifest_file, "recovery_manifest.json")

with zipfile.ZipFile(RECOVERY_ZIP, "r") as zf:
    bad = zf.testzip()
    assert bad is None, f"Recovery ZIP failed at: {bad}"
    print("\nRecovery bundle verified.")
    print("Files included:", len(zf.namelist()))

print("Recovery ZIP:", RECOVERY_ZIP)
print("Recovery ZIP size (MB):", round(RECOVERY_ZIP.stat().st_size / (1024**2), 2))


# ================= FINAL DURABLE DATASET PUBLISH =================
if DURABLE_PUBLISH_ENABLED:
    publish_recovery_dataset(reason="final_notebook_recovery")
else:
    print("\nFINAL DURABLE PUBLISH SKIPPED.")
    print("Configure Kaggle Secrets in Cell 0b to publish the backup.")
    print("Local recovery ZIP:", RECOVERY_ZIP)

print("\nAll prediction structure and ZIP validations passed.")
